# Overview: Decision Tree Families
Dataset: Breast Cancer Wisconsin (binary classification: malignant / benign)
Metrics: Accuracy + Classification Report

In [15]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Load dataset and split 80/20
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Classes: {load_breast_cancer().target_names}')

Train: (455, 30) | Test: (114, 30)
Classes: ['malignant' 'benign']


## 1. Single Tree
Model: DecisionTreeClassifier
A single decision tree. Does not combine anything. High variance, prone to overfitting.

In [16]:
from sklearn.tree import DecisionTreeClassifier

# max_depth=5 to prevent the tree from growing indefinitely
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print('=== Single Tree ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}')
print(classification_report(y_test, y_pred_dt, target_names=['malignant','benign']))

=== Single Tree ===
Accuracy: 0.9474
              precision    recall  f1-score   support

   malignant       0.93      0.93      0.93        43
      benign       0.96      0.96      0.96        71

    accuracy                           0.95       114
   macro avg       0.94      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114



## 2. Bagging
Model: BaggingClassifier
N trees trained in parallel with bootstrap sampling. Combines by majority voting. Reduces variance.

In [17]:
from sklearn.ensemble import BaggingClassifier

bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,   # number of trees
    max_samples=0.8,    # 80% of data per tree
    bootstrap=True,     # sampling with replacement
    random_state=42
)
bag.fit(X_train, y_train)
y_pred_bag = bag.predict(X_test)

print('=== Bagging ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_bag):.4f}')
print(classification_report(y_test, y_pred_bag, target_names=['malignant','benign']))

=== Bagging ===
Accuracy: 0.9561
              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        43
      benign       0.96      0.97      0.97        71

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



## 3. Random Forest
Model: RandomForestClassifier
Bagging plus a random subset of features at each split (diversity). Reduces more variance.

In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',  # uses sqrt(n_features) random features at each split
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== Random Forest ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(classification_report(y_test, y_pred_rf, target_names=['malignant','benign']))

=== Random Forest ===
Accuracy: 0.9649
              precision    recall  f1-score   support

   malignant       0.98      0.93      0.95        43
      benign       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



## 4. Boosting - AdaBoost
Model: AdaBoostClassifier
Sequential trees with sample weights: each tree corrects the errors of the previous one. Reduces bias.

In [19]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # weak learners (stumps)
    n_estimators=100,
    learning_rate=1.0,  # contribution of each tree to the final prediction
    random_state=42
)
ada.fit(X_train, y_train)
y_pred_ada = ada.predict(X_test)

print('=== AdaBoost ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_ada):.4f}')
print(classification_report(y_test, y_pred_ada, target_names=['malignant','benign']))

=== AdaBoost ===
Accuracy: 0.9737
              precision    recall  f1-score   support

   malignant       0.98      0.95      0.96        43
      benign       0.97      0.99      0.98        71

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



## 5. Gradient Boosting - XGBoost
Model: XGBClassifier
Sequential trees trained on residuals (gradient of the loss). Reduces bias. State of the art on tabular data.

In [20]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,       # gradient update step size
    subsample=0.8,           # fraction of rows per tree
    colsample_bytree=0.8,    # fraction of features per tree
    eval_metric='logloss',
    random_state=42
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print('=== XGBoost ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}')
print(classification_report(y_test, y_pred_xgb, target_names=['malignant','benign']))

=== XGBoost ===
Accuracy: 0.9649
              precision    recall  f1-score   support

   malignant       0.98      0.93      0.95        43
      benign       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



## Summary

In [21]:
import pandas as pd

results = {
    'Family':   ['Single Tree', 'Bagging', 'Random Forest', 'AdaBoost', 'XGBoost'],
    'Model':    ['DecisionTreeClassifier', 'BaggingClassifier', 'RandomForestClassifier', 'AdaBoostClassifier', 'XGBClassifier'],
    'Combines': ['none', 'Parallel + Voting', 'Parallel + Voting + Diversity', 'Sequential + Weights', 'Sequential + Residuals'],
    'Reduces':  ['Nothing', 'Variance', 'Variance (more)', 'Bias', 'Bias'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_bag),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_ada),
        accuracy_score(y_test, y_pred_xgb),
    ]
}

df = pd.DataFrame(results)
df['Accuracy'] = df['Accuracy'].map('{:.4f}'.format)
df

,Family,Model,Combines,Reduces,Accuracy
0,Single Tree,DecisionTreeClassifier,none,Nothing,0.9474
1,Bagging,BaggingClassifier,Parallel + Voting,Variance,0.9561
2,Random Forest,RandomForestClassifier,Parallel + Voting + Diversity,Variance (more),0.9649
3,AdaBoost,AdaBoostClassifier,Sequential + Weights,Bias,0.9737
4,XGBoost,XGBClassifier,Sequential + Residuals,Bias,0.9649
